# 02. Build Non-ICU RAAS Early Exposure Tables

Short summary:
[02 - Early RAAS Exposure Definition (Non-ICU Admissions)](../docs/02_exposure_SHORT.md)

## 1. Purpose

This notebook constructs early renin–angiotensin–aldosterone system (RAAS) inhibitor exposure variables for adult, non-ICU hospital admissions using the MIMIC-IV database.

Using SQL-based preprocessing in BigQuery, inpatient medication prescription records are linked to hospital admission timestamps to identify early-in-admission RAAS inhibitor exposure, defined as at least one prescription for an angiotensin-converting enzyme inhibitor (ACE inhibitor) or angiotensin II receptor blocker (ARB) with `admittime <= starttime < admittime + 24 hours` (0–<24 hours). This is a prescription-record indicator and does not confirm administration or adherence.

The resulting exposure indicators are constructed at the hospital admission (`hadm_id`) level and materialized as BigQuery tables for use in downstream cohort description and outcome analyses.

### 1.1 Exposure Definition Task

The purpose of this notebook is to define and construct early RAAS inhibitor exposure variables at the hospital admission level for adult, non-ICU admissions in MIMIC-IV.

Specifically, this notebook:
- Defines early RAAS inhibitor exposure using inpatient prescription records from the MIMIC-IV prescriptions table
- Restricts recorded prescription start times to 0–<24 hours after admission; the original full-admission analysis includes early deaths and this window alone does not establish a fixed prediction time or eliminate time-related bias
- Constructs admission-level binary indicators for:
  - ACE inhibitor exposure
  - ARB exposure
  - Prescription records from both classes within the early window; no overlap or administration requirement
  - Any RAAS inhibitor exposure
- Materializes these exposure variables as reproducible BigQuery tables for downstream baseline characterization and outcome modeling

This notebook focuses exclusively on exposure variable construction and does not perform descriptive analyses, statistical modeling, or outcome evaluation.

## 2. Inputs

- **MIMIC-IV v3.1** (BigQuery public dataset)
- Project: `mimic-iv-portfolio`

**Source Tables:**
- `physionet-data.mimiciv_3_1_hosp.prescriptions`<br>
  (BigQuery public dataset)

- `mimic-iv-portfolio.nonicu_raas.nonicu_admissions`<br>
  (created in [01_cohort.ipynb](01_cohort.ipynb) using [02_exclude_icu_admissions.sql](../sql/02_exclude_icu_admissions.sql)

- `mimic-iv-portfolio.nonicu_raas.exposure_raas_early`<br>
  (created in 02_exposure.ipynb using [03_define_exposure_raas_early.sql](../sql/03_define_exposure_raas_early.sql)

**Derived datasets created in this notebook:**
- `mimic-iv-portfolio.nonicu_raas.exposure_raas_early`
- `mimic-iv-portfolio.nonicu_raas.analysis_dataset`

## 3. Methods
### 3.1 BigQuery Client Initialization and Helper Functions

## Reading the results

Data sources and execution mode are summarized once below, after the destination settings. Expand **Calculation inputs and source definitions** to trace individual SQL definitions and aggregate references.

Each result is generated by Python in this order: **number/title → figure or aggregate table → how to read → computed result summary → interpretation limits**. Cohort definitions, exposures, adjustment variables and estimators are unchanged. See the [reproduction instructions](../REPRODUCIBILITY.md#python-generated-notebook-results).


The input-definition index describes the Python objects/queries used in calculation. The separate **Saved aggregates and verification files (CSV/JSON)** list identifies saved results and checking records, rather than original MIMIC-IV source tables. Live notebook displays use in-memory results; saved-output replay and validation workflows may read these files.

**Live input behavior:** Normal execution requires the configured real BigQuery inputs and approved access. Missing settings, credentials, permissions or tables stop execution with a cause and a next step; no synthetic/demo data or saved results are substituted. Stored presentation-replay outputs are review artifacts, not a successful new data load. Use a fresh kernel for a new run.


In [ ]:
# Presentation uses only objects produced by this notebook execution.
from pathlib import Path as _ResultPath
import sys as _result_sys
_result_root = _ResultPath.cwd().resolve()
if _result_root.name == "notebooks":
    _result_root = _result_root.parent
if not (_result_root / "scripts" / "nonicu_notebook_results.py").is_file():
    raise RuntimeError("Run this notebook from the repository root or its notebooks directory")
if str(_result_root / "scripts") not in _result_sys.path:
    _result_sys.path.insert(0, str(_result_root / "scripts"))
from nonicu_notebook_results import NotebookResults
notebook_results = NotebookResults(globals())


In [ ]:
from real_data_access import real_data_access, real_data_io
# Use Application Default Credentials (my user account)
# Use your own approved ADC account; access is checked during execution.

from google.cloud import bigquery
from google.auth import default
from pathlib import Path

# 1. Define project ID, dataset, and table references
PROJECT_ID = "mimic-iv-portfolio"
DATASET = "nonicu_raas"

TABLE_NONICU = f"{PROJECT_ID}.{DATASET}.nonicu_admissions"   # Created in 02_exclude_icu_admissions.sql
TABLE_EXPO = f"{PROJECT_ID}.{DATASET}.exposure_raas_early"   # Created in 03_define_exposure_raas_early.sql
TABLE_ANALYSIS = f"{PROJECT_ID}.{DATASET}.analysis_dataset"  # Created in 04_build_analysis_dataset.sql


# 2. Get ADC credentials
with real_data_access(PROJECT_ID, DATASET):
    creds, adc_project = default()

    # 3. Initialize BigQuery client
    client = bigquery.Client(project=PROJECT_ID, credentials=creds)

# 5. Helper to run a SQL script file (DDL, CREATE TABLE, etc.)
@real_data_io
def run_sql_script(path) :
    """
    Read a .sql file from disk, execute it in BigQuery,
    and wait until the job finishes.
    Use this for CREATE TABLE / INSERT INTO scripts.
    """
    sql_path = Path(path)
    with sql_path.open("r") as f:
        query = f.read()
    job = client.query(query)
    job.result()
    print(f"Executed SQL script: {sql_path.name}")

# 6. Helper for read-only SELECT queries → DataFrame
@real_data_io
def query_to_df(query) :
    """
    Run a SELECT query in BigQuery and return a pandas DataFrame.
    """
    job = client.query(query)
    return job.to_dataframe(create_bqstorage_client=False)

In [ ]:
# Shared source references and execution mode for all numbered results.
notebook_results.overview(['Table 02.1', 'Table 02.2', 'Table 02.3', 'Table 02.4', 'Table 02.5', 'Table 02.6', 'Table 02.7'])


### 3.2 Execute SQL to Define Early RAAS Exposure

This step defines early inpatient exposure to RAAS inhibitors (ACE inhibitors and ARBs) within the first 24 hours of hospital admission in order to construct an exposure variable aligned with downstream mortality analyses in a non-ICU adult inpatient cohort.

- SQL script: [03_define_exposure_raas_early.sql](../sql/03_define_exposure_raas_early.sql)
- Output table: `nonicu_raas.exposure_raas_early`

The SQL script identifies RAAS inhibitor medication orders using the `physionet-data.mimiciv_3_1_hosp.prescriptions` table, which contains inpatient prescription records with medication names and start times. RAAS inhibitors are captured by matching prescription drug names to a pre-specified list of ACE inhibitors and ARBs (and/or by keyword-based mapping rules defined in the script).

Exposure ascertainment uses `admittime <= starttime < admittime + 24 hours`. A recorded start time does not establish when the order was entered or confirm actual administration. The original full-admission cohort includes early deaths; the early window alone does not eliminate reverse causation or immortal-time concerns. The corrected 24-hour association sensitivity in 04b applies its separate survival/continued-hospitalization eligibility filter.

Admissions are classified as RAAS inhibitor exposed if at least one qualifying ACE inhibitor or ARB prescription is observed in this early window. The script additionally supports subclass categorization by deriving indicators for ACE inhibitor exposure and ARB exposure separately, enabling comparisons of overall RAAS exposure as well as ACEi vs ARB subgroup analyses.

The resulting table provides an admission-level exposure definition that can be joined back to the non-ICU cohort, supporting both relative effect estimation (e.g., odds ratios) and absolute risk estimation (e.g., marginal effects) in downstream models focused on general ward (non-ICU) hospitalized populations.

In [ ]:
# Excecute SQL script (03_define_exposure_raas_early.sql)
run_sql_script("../sql/03_define_exposure_raas_early.sql")

### 3.3 Load and Inspect Early RAAS Exposure Table

In [ ]:
with notebook_results.capture('Table 02.1', globals()):
    q_expo = f"""
SELECT *
FROM `{TABLE_EXPO}`
"""
    df_expo = client.query(q_expo).to_dataframe()

    df_expo.info()
    print("Shape:", df_expo.shape)
    _notebook_result_value = (df_expo["raas_any_early"].value_counts(dropna=False))


### 3.4 Build Final Admission-Level Analysis Dataset

This step constructs the final analytic dataset by combining the non-ICU admission cohort with early RAAS inhibitor exposure variables at the hospital admission (HADM) level.

- SQL script: [03_build_analysis_dataset.sql](../sql/03_build_analysis_dataset.sql)
- Output table: `nonicu_raas.analysis_dataset`

The SQL script starts from the non-ICU adult inpatient cohort stored in `nonicu_raas.nonicu_admissions`, which contains admission-level and patient-level variables for hospitalizations without any ICU stay.

Early RAAS inhibitor exposure variables are sourced from the nonicu_raas.exposure_raas_early table, which defines ACE inhibitor and ARB exposure within the first 24 hours of hospital admission.

These two tables are linked using a left join on the shared identifiers `subject_id` and `hadm_id`, ensuring that all admissions in the non-ICU cohort are retained regardless of RAAS exposure status.

Exposure indicators are incorporated into the analytic dataset using COALESCE to assign a value of 0 to admissions without qualifying prescriptions. The resulting variables include separate indicators for early ACE inhibitor exposure (`acei_early`), early ARB exposure (`arb_early`), exposure to both drug classes (`raas_both_early`), and exposure to either class (`raas_any_early`).

The resulting table represents the complete HADM-level analytic dataset used for downstream descriptive analyses and multivariable outcome modeling, integrating cohort definition, baseline characteristics, and early medication exposure in a single, analysis-ready structure.

In [ ]:
# Excecute SQL script (03_build_analysis_dataset.sql)
run_sql_script("../sql/03_build_analysis_dataset.sql")

### 3.5 Load and Inspect Final Analysis Dataset

In [ ]:
with notebook_results.capture('Table 02.2', globals()):
    q_analysis = f"""
SELECT *
FROM `{TABLE_ANALYSIS}`
"""
    df = client.query(q_analysis).to_dataframe()

    df.info()
    print("Shape:", df.shape)
    _notebook_result_value = (df["raas_any_early"].value_counts(dropna=False))


## 4. Quality Checks

### 4.1 Exposure Indicator Validation and Prevalence

In [ ]:
with notebook_results.capture('Table 02.3', globals()):
    consistency_query = f"""
SELECT
  AVG(CAST(((acei_early = 1 OR arb_early = 1) = (raas_any_early = 1)) AS INT64)) AS any_consistency,
  AVG(CAST(((acei_early = 1 AND arb_early = 1) = (raas_both_early = 1)) AS INT64)) AS both_consistency
FROM `{TABLE_EXPO}`
"""

    _notebook_result_value = (client.query(consistency_query).to_dataframe())


In [ ]:
with notebook_results.capture('Table 02.4', globals()):
    # Identify and count any records where logical inconsistencies are present.
    mismatch_query = f"""
SELECT
  COUNTIF((acei_early = 1 OR arb_early = 1) != (raas_any_early = 1)) AS any_mismatches,
  COUNTIF((acei_early = 1 AND arb_early = 1) != (raas_both_early = 1)) AS both_mismatches
FROM `{TABLE_EXPO}`
"""

    _notebook_result_value = (client.query(mismatch_query).to_dataframe())


### 4.2 Row Count Comparison Between Cohort and Exposure Tables

In [ ]:
with notebook_results.capture('Table 02.5', globals()):
    q_counts = f"""
SELECT
  (SELECT COUNT(*) FROM `{TABLE_NONICU}`) AS n_nonicu,
  (SELECT COUNT(*) FROM `{TABLE_EXPO}`) AS n_expo
"""
    _notebook_result_value = (client.query(q_counts).to_dataframe())


### 4.3 Distribution of Matched RAAS Drugs

This step inspects the specific medication names contributing to early RAAS inhibitor exposure in the non-ICU admission cohort, serving as a validation and transparency check for the exposure definition.

- SQL script: [04_check_raas_drugs.sql](../sql/04_check_raas_drugs.sql)
- Output: summary query result (no table created)

The SQL script identifies all inpatient medication prescriptions initiated within the first 24 hours after hospital admission among non-ICU admissions stored in `nonicu_raas.nonicu_admissions`. Prescription records are sourced from the `physionet-data.mimiciv_3_1_hosp.prescriptions` table and linked to admissions using `subject_id` and `hadm_id`.

Medication names are converted to lowercase to standardize string matching, and prescriptions with missing drug names are excluded. The resulting set represents documented inpatient prescription exposure during the early admission window; it does not confirm medication administration or adherence.

Each medication is then classified using regular expression matching into angiotensin-converting enzyme inhibitors (ACEi), angiotensin II receptor blockers (ARB), or other drugs based on the drug name text. Only medications classified as ACEi or ARB are retained in the final summary.

The query aggregates counts of prescription records by drug class and individual drug name, producing a frequency table ordered by descending count. This output allows direct inspection of which specific ACE inhibitors and ARBs drive the early exposure definition and supports verification that the exposure algorithm captures clinically appropriate medications while excluding unrelated drugs.

This step does not modify the analytic dataset but provides an important diagnostic check to ensure the validity and interpretability of the RAAS exposure definition used in downstream analyses.

In [ ]:
with notebook_results.capture('Table 02.6', globals()):
    df_drugs = query_to_df(
        Path("../sql/04_check_raas_drugs.sql").read_text()
    )

    _notebook_result_value = (df_drugs)


### 4.4 Missingness of Prescription Start Times

In [ ]:
with notebook_results.capture('Table 02.7', globals()):
    query="""
SELECT
  COUNT(*) AS n_total,
  SUM(CASE WHEN starttime IS NULL THEN 1 ELSE 0 END) AS n_starttime_null,
  SUM(CASE WHEN starttime IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_starttime_null
FROM `physionet-data.mimiciv_3_1_hosp.prescriptions`
WHERE hadm_id IS NOT NULL
  AND drug IS NOT NULL;
"""
    _notebook_result_value = (client.query(query).to_dataframe())


## 5. Outputs

This notebook produces and validates admission-level tables in BigQuery that are used as fixed inputs for the original association branch (03a–04b).

**Outputs**

As a result of executing the SQL scripts in this notebook, the following tables are materialized in the
mimic-iv-portfolio.nonicu_raas dataset:

- **exposure_raas_early**
An admission-level table encoding early RAAS inhibitor exposure, defined by ACE inhibitor or ARB prescription records with `admittime <= starttime < admittime + 24 hours`. Exposure variables are stored as binary indicators (`acei_early`, `arb_early`, `raas_both_early`, `raas_any_early`).

- **analysis_dataset**
A unified, HADM-level analytic dataset created by left-joining the non-ICU admissions cohort with the early RAAS exposure table on (`subject_id`, `hadm_id`).
Each row corresponds to a single hospital admission, with exposure indicators explicitly coded for all admissions using zero-filling for non-exposed cases.

These tables are constructed entirely via SQL and are treated as finalized outputs of the data preparation stage.

**Downstream Use**

The `analysis_dataset` table supplies descriptive and outcome analyses in 03b–04b; 03a separately reads the component cohort and exposure tables. These steps include:

- Baseline characteristic summaries ([03a_validate_input_tables.ipynb](03a_validate_input_tables.ipynb) and [03b_describe_analysis_dataset.ipynb](03b_describe_analysis_dataset.ipynb))
- Unadjusted outcome summaries and crude analyses ([04a_unadjusted_outcomes.ipynb](04a_unadjusted_outcomes.ipynb))
- Multivariable and adjusted outcome modeling ([04b_multivariable_outcomes.ipynb](04b_multivariable_outcomes.ipynb))

Downstream association notebooks read these tables without rewriting them or changing the exposure definition. Notebook 04b separately applies its documented landmark eligibility filters. Both-class indicators mean records from both drug classes within the window, not overlapping prescriptions or simultaneous administration. A zero flag denotes no qualifying record, not proven absence of medication use.

---
## Next Step

Proceed to:
- [03a. Input Table Validation and Descriptive Summary](03a_validate_input_tables.ipynb)

This next notebook/document briefly describes the next workflow step.

**Separate 04c feature:** [04c](04c_prediction_time_evaluation.ipynb) starts from the adult base and native location/order records. Its 24-hour companion requires both an early prescription start and a linked POE order time by 24 hours. It does not reuse the association flag as proof of information available at prediction time. See the [04c protocol](../docs/PREDICTION_TIME_PROTOCOL.md).
